# 10회차 실습 노트북 — 회귀모형(2) — 로지스틱·결정나무·특성공학

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/10_logistic_tree.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/10_logistic_tree.html](https://adminmrobo.github.io/fintech_data_analysis/10_logistic_tree.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 로지스틱 곡선: 거래 횟수로 이탈 확률

In [2]:
b = load('Revolution/Bank_customers_data.csv')
b['y'] = (b.Attrition_Flag == 'Attrited Customer').astype(int)   # 1 = 이탈 고객
print(b.shape, '이탈 비율:', round(b.y.mean(), 3))
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression().fit(b[['Total_Trans_Ct']], b.y)
xs = np.arange(10, 140).reshape(-1, 1)
plt.plot(xs, lr.predict_proba(xs)[:, 1], label='logistic')
g = b.groupby(pd.cut(b.Total_Trans_Ct, range(10, 141, 10))).y.mean()
plt.plot([iv.mid for iv in g.index], g.values, 'o', label='actual (10-wide bins)')
plt.xlabel('Total_Trans_Ct'); plt.ylabel('P(attrition)'); plt.legend(); plt.show()

(10127, 22) 이탈 비율: 0.161


실제 비율(점)은 40~50회 구간에서 다시 튀어 오릅니다. S자 곡선 하나로는 이 모양을 못 따라가요.

## 2. 결정나무: 깊이에 따른 학습·시험 점수

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
X = b[['Total_Trans_Ct', 'Total_Revolving_Bal']].values.astype(float)
Xtr, Xte, ytr, yte = train_test_split(X, b.y, test_size=.25, random_state=RS, stratify=b.y)
for d in [1, 2, 3, 4, 6, 10, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=RS).fit(Xtr, ytr)
    print(f'깊이 {str(d):4s} 학습 {m.score(Xtr, ytr):.3f}  시험 {m.score(Xte, yte):.3f}  잎 {m.get_n_leaves()}')
print(export_text(DecisionTreeClassifier(max_depth=2, random_state=RS).fit(Xtr, ytr), feature_names=['Trans_Ct', 'Revolving_Bal']))

깊이 1    학습 0.839  시험 0.839  잎 2
깊이 2    학습 0.894  시험 0.887  잎 4
깊이 3    학습 0.895  시험 0.883  잎 8
깊이 4    학습 0.898  시험 0.890  잎 16
깊이 6    학습 0.901  시험 0.892  잎 62
깊이 10   학습 0.914  시험 0.882  잎 271
깊이 None 학습 0.946  시험 0.857  잎 707
|--- Trans_Ct <= 54.50
|   |--- Revolving_Bal <= 615.50
|   |   |--- class: 1
|   |--- Revolving_Bal >  615.50
|   |   |--- class: 0
|--- Trans_Ct >  54.50
|   |--- Revolving_Bal <= 594.50
|   |   |--- class: 0
|   |--- Revolving_Bal >  594.50
|   |   |--- class: 0



## 3. 대결 재현: 기본 특성 3개 vs 이유 있게 만든 특성 6개

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
ins = load('Revolution/insurance.csv')
ins['sm'] = (ins.smoker == 'yes').astype(int); ins['obese'] = (ins.bmi >= 30).astype(int)
ins['age2'] = ins.age ** 2; ins['sm_obese'] = ins.sm * ins.obese; ins['sm_bmi'] = ins.sm * ins.bmi
tr, te = train_test_split(ins, test_size=.25, random_state=RS)
for cols in (['age', 'bmi', 'sm'], ['age', 'age2', 'bmi', 'sm', 'sm_obese', 'sm_bmi']):
    m = LinearRegression().fit(tr[cols], tr.charges)
    print(len(cols), '개 특성  R² =', round(r2_score(te.charges, m.predict(te[cols])), 3))

3 개 특성  R² = 0.762
6 개 특성  R² = 0.865


**AI 판단 연습** — "흡연 × 비만"이라는 특성은 왜 의미가 있을까요? 데이터 산점도(흡연자만 BMI 30을 넘으면 보험료가 뜀)로 확인해 보세요.